# 99 — Reset: limpando o banco (opcional)

Notebook utilitário, fora da sequência do material. Ele **apaga tudo** do seu
banco Neo4j: todos os nós, todos os relacionamentos, todas as constraints e todos
os índices.

Quando isso é útil:

- Você quer recomeçar o material do zero
- Mudou os parâmetros do notebook 01 e quer recarregar sem misturar com os dados antigos
- Alguma carga deu errado no meio e você quer partir de um estado limpo
- Vai apresentar o material e quer o banco vazio antes de começar

> ⚠️ **Isto é irreversível.** Não há lixeira, não há undo. Só rode se tiver certeza
> de que os dados deste banco podem ser perdidos — e confira, na célula de
> conexão, que você está apontando para a instância certa.
>
> Por segurança, a célula que apaga só funciona depois que você trocar
> `CONFIRMAR = False` por `CONFIRMAR = True`.

In [ ]:
!pip install -q neo4j-rust-ext python-dotenv

## Credenciais

Este notebook busca as credenciais em três lugares, na ordem:

1. **Variáveis de ambiente** — incluindo um arquivo `.env` na pasta do projeto
   (copie o `.env.example` e preencha). É o jeito recomendado: você configura uma
   vez e todos os notebooks funcionam sem digitar nada.
2. **Secrets do Colab** — o ícone de chave 🔑 na barra lateral esquerda. Crie um
   secret por variável (`NEO4J_URI`, `NEO4J_PASSWORD`, …) e ative o acesso para
   este notebook.
3. **Pergunta na tela** — se não achou nas opções acima, pergunta aqui mesmo.

Além de poupar digitação, as duas primeiras opções evitam que a URI da sua
instância fique gravada na saída da célula caso você comite o notebook.

In [ ]:
import os
from getpass import getpass

try:  # carrega um arquivo .env, se existir
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass  # python-dotenv não instalado, ou não há .env — segue o baile


def credencial(nome, prompt, secreta=False, padrao=None):
    """Busca em: variável de ambiente > Secrets do Colab > pergunta na tela."""
    if valor := os.environ.get(nome):
        return valor
    try:
        from google.colab import userdata
        if valor := userdata.get(nome):
            return valor
    except Exception:
        pass  # não é Colab, ou o secret não existe/não foi liberado
    return (getpass(prompt) if secreta else input(prompt)).strip() or padrao


NEO4J_URI = credencial("NEO4J_URI", "URI do Neo4j (ex.: neo4j+s://xxxx.databases.neo4j.io): ")
NEO4J_USER = credencial("NEO4J_USERNAME", "Usuário [neo4j]: ", padrao="neo4j")
NEO4J_PASSWORD = credencial("NEO4J_PASSWORD", "Senha: ", secreta=True)
# Atenção: em instâncias AuraDB recentes o banco NÃO se chama "neo4j", e sim o
# próprio instance id (o prefixo da URI). Confira em Aura Console > sua instância,
# ou rode SHOW DATABASES.
NEO4J_DATABASE = credencial("NEO4J_DATABASE", "Nome do banco (geralmente = instance id) [neo4j]: ", padrao="neo4j")

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Conectado!")

## 1. O que existe hoje no banco

Antes de apagar, veja o que você tem. Confira também a URI impressa: é a última
chance de perceber que você conectou na instância errada.

In [ ]:
print(f"Instância: {NEO4J_URI}")
print(f"Banco:     {NEO4J_DATABASE}\n")

records, _, _ = driver.execute_query(
    "MATCH (n) RETURN count(n) AS nos", database_=NEO4J_DATABASE
)
total_nos = records[0]["nos"]

records, _, _ = driver.execute_query(
    "MATCH ()-[r]->() RETURN count(r) AS rels", database_=NEO4J_DATABASE
)
total_rels = records[0]["rels"]

records, _, _ = driver.execute_query(
    "SHOW CONSTRAINTS YIELD name RETURN collect(name) AS nomes", database_=NEO4J_DATABASE
)
constraints = records[0]["nomes"]

records, _, _ = driver.execute_query(
    """
    SHOW INDEXES YIELD name, type
    WHERE type <> 'LOOKUP'
    RETURN collect(name) AS nomes
    """,
    database_=NEO4J_DATABASE,
)
indices = records[0]["nomes"]

print(f"Nós:            {total_nos:,}")
print(f"Relacionamentos:{total_rels:>10,}")
print(f"Constraints:    {len(constraints)}  {constraints}")
print(f"Índices:        {len(indices)}  {indices}")

## 2. Confirmação

Troque para `True` e rode a célula. Deixei o padrão em `False` de propósito: assim
um "rodar todas as células" sem pensar não apaga o seu banco.

In [ ]:
CONFIRMAR = False  # <<< mude para True para apagar de verdade

## 3. Apagando os dados

Apagamos em lotes de 10.000 nós, e não tudo de uma vez. O motivo: um
`MATCH (n) DETACH DELETE n` num banco grande monta uma transação gigante na
memória e pode falhar — justamente no AuraDB Free, onde a memória é mais
apertada. Em lotes, cada transação é pequena e o progresso é visível.

`DETACH DELETE` remove o nó **e** os relacionamentos ligados a ele; um `DELETE`
puro falharia em qualquer nó que ainda tivesse relacionamento.

In [ ]:
if not CONFIRMAR:
    print("Nada foi apagado. Mude CONFIRMAR para True na célula acima se quiser prosseguir.")
else:
    total_apagado = 0
    while True:
        records, _, _ = driver.execute_query(
            """
            MATCH (n)
            WITH n LIMIT 10000
            DETACH DELETE n
            RETURN count(n) AS apagados
            """,
            database_=NEO4J_DATABASE,
        )
        apagados = records[0]["apagados"]
        if apagados == 0:
            break
        total_apagado += apagados
        print(f"  {total_apagado:,} nós apagados...")

    print(f"\nPronto: {total_apagado:,} nós removidos.")

## 4. Removendo constraints e índices

A ordem importa: cada constraint de unicidade tem um índice por trás, e remover a
constraint já remove esse índice. Então tratamos as constraints primeiro e, depois,
os índices que sobraram (os que você criou à mão, se criou).

Índices do tipo `LOOKUP` são internos do Neo4j e ficam de fora — eles não guardam
dados seus e o banco os recria de qualquer forma.

In [ ]:
if not CONFIRMAR:
    print("Nada foi removido (CONFIRMAR está False).")
else:
    records, _, _ = driver.execute_query(
        "SHOW CONSTRAINTS YIELD name RETURN name", database_=NEO4J_DATABASE
    )
    for r in records:
        driver.execute_query(f"DROP CONSTRAINT {r['name']}", database_=NEO4J_DATABASE)
        print(f"  constraint removida: {r['name']}")

    records, _, _ = driver.execute_query(
        """
        SHOW INDEXES YIELD name, type
        WHERE type <> 'LOOKUP'
        RETURN name
        """,
        database_=NEO4J_DATABASE,
    )
    for r in records:
        driver.execute_query(f"DROP INDEX {r['name']}", database_=NEO4J_DATABASE)
        print(f"  índice removido: {r['name']}")

    print("\nSchema limpo.")

## 5. Conferindo

Tudo em zero significa que o banco está como estava quando você o criou.

In [ ]:
# Consultas separadas de propósito: um MATCH encadeado não retornaria linha
# nenhuma quando a contagem intermediária fosse zero.
records_nos, _, _ = driver.execute_query("MATCH (n) RETURN count(n) AS nos", database_=NEO4J_DATABASE)
records_rels, _, _ = driver.execute_query("MATCH ()-[r]->() RETURN count(r) AS rels", database_=NEO4J_DATABASE)
records_con, _, _ = driver.execute_query("SHOW CONSTRAINTS YIELD name RETURN count(*) AS c", database_=NEO4J_DATABASE)

print(f"Nós:             {records_nos[0]['nos']}")
print(f"Relacionamentos: {records_rels[0]['rels']}")
print(f"Constraints:     {records_con[0]['c']}")

if records_nos[0]["nos"] == 0 and records_con[0]["c"] == 0:
    print("\n✅ Banco vazio. Pode recomeçar do notebook 01.")
else:
    print("\nAinda há coisas no banco — rode as células 3 e 4 com CONFIRMAR = True.")

## Sobre as sessões do Aura Graph Analytics

Este notebook limpa o **banco de dados**, mas não as sessões de análise do Aura
Graph Analytics (elas vivem fora da instância). O notebook 04 já encerra a sessão
que cria, no fim da seção 5.

Se uma execução foi interrompida no meio e você desconfia que ficou uma sessão
aberta, liste e remova assim (precisa das credenciais de API):

```python
from graphdatascience.session import GdsSessions, AuraAPICredentials

sessions = GdsSessions(api_credentials=AuraAPICredentials(CLIENT_ID, CLIENT_SECRET))
print(sessions.list())
sessions.delete(session_name="fraude-workshop")
```

Sessões também expiram sozinhas pelo TTL (30 minutos no tier Free), e sessões nos
tiers Free e Pro Trial não são cobradas.

In [ ]:
driver.close()